# S1 Equities Factor Panel

Builds a **long-format factor panel** for the S1 equities sleeve: daily OHLCV plus engineered alpha features on a point-in-time tradable universe.

**Output:** a parquet file in this directory that downstream notebooks use to screen factors (IC, quintile spreads, Alphalens) and, later, to feed the equities backtest with the same signal timing and universe rules.

**Integrity constraints:** features at date `t` use only information available at or before `t`; universe membership is point-in-time (no survivorship bias); forward returns live in separate label columns for training/evaluation only.

Factor hypotheses under test are logged in `02_research/hypothesis_log.md`.

### What is a panel?

In this notebook, **panel** means a pandas **DataFrame** — the same tabular object you would normally call a dataframe. We use *panel* when the table has a repeated cross-section over time: one row per `(date, ticker)` with columns for prices, factors, and labels.

That layout is also called **long format** (stacked dates and tickers) as opposed to **wide format** (dates as rows, tickers as columns), which some libraries use only at export time. Throughout this notebook, `panel` is just the variable name for the working DataFrame.

## 0. Imports & Config

Project paths, cache locations, date range, universe size, and parquet output path.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

from data.ingestion.equity_fetcher import DEFAULT_CACHE_DIR, fetch_ohlcv, fetch_top_n_equities
from data.processing.feature_implementation.beta import add_rolling_beta, market_return_frame

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
# ROOT is then the trading_portfolio directory (used for project-wide paths like ALT_DATA_CACHE_DIR).
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
    
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
OUTPUT_PATH = os.path.join(NOTEBOOK_DIR, "s1_factor_panel.parquet")
ALT_DATA_CACHE_DIR = os.path.join(ROOT, "01_data", "cache", "alternative_data")

START_DATE = "2020-01-02"
END_DATE = None
UNIVERSE_N = 100
LOOKBACK_DAYS = 20
BETA_WINDOWS = [20]  # ints only; add e.g. 60 later — each window gets its own column set

### Survivorship bias and yfinance download failures

`fetch_top_n_equities` ranks a **point-in-time** S&P 500 universe, then downloads OHLCV via yfinance. You may see warnings such as `possibly delisted; no timezone found` or `no price data found` for names that were in the universe on the panel start date but are no longer available from Yahoo.

**What this means**

- **Delisted / acquired tickers** (e.g. ATVI, XLNX): Yahoo often removes history. Those names can rank in the top-N by dollar volume but are silently dropped from the downloaded panel — a **survivor bias** toward names Yahoo still serves.
- **Symbol-format mismatches** (e.g. `BRK.B`, `BF.B`): fixed in `equity_fetcher` by mapping canonical `.` tickers to Yahoo `-` symbols and mapping back on output.
- **Transient Yahoo errors** (e.g. BK): can poison the ranking cache if the empty result is cached; delete affected cache files under `01_data/cache/` and re-run.

Ticker mapping fixes symbol-format issues but **does not** restore delisted history. Treat the panel as potentially survivor-biased until a PIT-capable vendor replaces yfinance for dead names.


## 1. Data Loading

Fetch or load daily OHLCV for the point-in-time S&P 500 universe (top-N by trailing dollar volume as of the panel start).

In [ ]:
panel = fetch_top_n_equities(
    n=UNIVERSE_N,
    start_date=START_DATE,
    end_date=END_DATE,
    lookback_days=LOOKBACK_DAYS,
    cache_dir=DEFAULT_CACHE_DIR,
)

print(f"shape: {panel.shape}")
print(f"tickers: {panel['ticker'].nunique()}")
print(f"date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
panel.head()

$ABMD: possibly delisted; no timezone found
$ARNC: possibly delisted; no timezone found
$ABC: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$ALXN: possibly delisted; no timezone found
$ADS: possibly delisted; no timezone found
$ANTM: possibly delisted; no timezone found
$ANSS: possibly delisted; no timezone found

8 Failed downloads:
['ABMD', 'ARNC', 'ABC', 'AGN', 'ALXN', 'ADS', 'ANTM', 'ANSS']: possibly delisted; no timezone found
$ATVI: possibly delisted; no timezone found
$CMA: possibly delisted; no timezone found
$BF.B: possibly delisted; no price data found  (1d 2019-12-05 -> 2020-01-03)
$CERN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2019-12-05 -> 2020-01-03) (Yahoo error = "Data doesn't exist for startDate = 1575522000, endDate = 1578027600")
$BLL: possibly delisted; no timezone found
$BRK.B: possibly delisted; no timezone found

7 Failed downloads:
['ATVI', 'CMA', 'CERN', 'BLL', 'BRK.B']: possibly de

shape: (164199, 7)
tickers: 100
date range: 2020-01-02 → 2026-07-16


Price,date,ticker,open,high,low,close,volume
0,2020-01-02,AAPL,71.344062,72.394093,71.091191,72.333885,135480400.0
1,2020-01-02,ABBV,67.852737,68.225971,67.418565,68.210739,5639200.0
2,2020-01-02,ABT,75.995184,76.789930,75.765597,76.781097,4969000.0
3,2020-01-02,ACN,189.265299,190.216740,187.425242,188.628006,2431100.0
4,2020-01-02,ADBE,330.000000,334.480011,329.170013,334.429993,1990100.0


## 2. Integration of Alternative Data

Merge cached alternative-data features (e.g. news sentiment) onto the OHLCV panel. Align with `merge_asof(..., direction='backward')` so each row at date `t` only sees data published on or before `t`.

### 2.1 Beta

Rolling OLS of each stock's daily log returns on SPY, estimated **per ticker** over every window in `BETA_WINDOWS`. Features at date `t` use returns through the close of `t` only (`min_periods=W`; partial windows → NaN). No `bfill` on returns or regression outputs.

**Model:** `r_{i,t} = alpha + beta * r_{SPY,t} + epsilon_t`

**Benchmark:** SPY via `fetch_ohlcv("SPY", START_DATE, END_DATE, cache_dir=DEFAULT_CACHE_DIR)`.

**Returns:** `ln(close_t / close_{t-1})`; the first bar of each series is NaN.

**Attached columns** (alpha / beta / r2 only — idio vol is built later in H-003):

| Column naming | When |
|---------------|------|
| `alpha`, `beta`, `r2` | `len(BETA_WINDOWS) == 1` |
| `alpha_{W}`, `beta_{W}`, `r2_{W}` | `len(BETA_WINDOWS) > 1` |

**Not stored here:** `idio_vol` (residual std) and per-day residuals `epsilon_t`. Idiosyncratic vol is implemented in `data.processing.feature_implementation.idiosyncratic_vol` and tested in `02_research/notebooks/factor_tests/H-003_idiosyncratic_vol.ipynb`.

Uses `add_rolling_beta` from `data.processing.feature_implementation.beta` for reproducibility in walk-forward and live pipelines. Supports H-004 (beta); H-003 factor engineering is deferred.


In [5]:
spy_panel = fetch_ohlcv("SPY", START_DATE, END_DATE, cache_dir=DEFAULT_CACHE_DIR)
spy_ret = market_return_frame(spy_panel, out_col="spy_log_ret")

panel = add_rolling_beta(
    panel,
    spy_ret.rename(columns={"spy_log_ret": "market_log_ret"}),
    windows=BETA_WINDOWS,
    market_col="market_log_ret",
)

beta_cols = [c for c in panel.columns if c in ("alpha", "beta", "r2") or c.startswith(("alpha_", "beta_", "r2_"))]
print(f"beta columns: {beta_cols}")
print(panel[beta_cols].describe())
panel.head()


beta columns: ['alpha_20', 'beta_20', 'idio_vol_20', 'r2_20']
            alpha_20        beta_20    idio_vol_20         r2_20
count  162199.000000  162199.000000  162199.000000  1.621990e+05
mean       -0.000105       0.909630       0.015263  3.152266e-01
std         0.004021       0.733044       0.008254  2.481657e-01
min        -0.044101      -3.502796       0.002738  9.947598e-14
25%        -0.002290       0.482473       0.010080  9.031556e-02
50%        -0.000056       0.875019       0.013364  2.754090e-01
75%         0.002169       1.285358       0.018031  5.037052e-01
max         0.028756       7.741018       0.157032  9.677335e-01


,date,ticker,open,high,low,close,volume,alpha_20,beta_20,idio_vol_20,r2_20
0,2020-01-02,AAPL,71.344062,72.394093,71.091191,72.333885,135480400.0,NaN,NaN,NaN,NaN
1,2020-01-02,ABBV,67.852737,68.225971,67.418565,68.210739,5639200.0,NaN,NaN,NaN,NaN
2,2020-01-02,ABT,75.995184,76.789930,75.765597,76.781097,4969000.0,NaN,NaN,NaN,NaN
3,2020-01-02,ACN,189.265299,190.216740,187.425242,188.628006,2431100.0,NaN,NaN,NaN,NaN
4,2020-01-02,ADBE,330.000000,334.480011,329.170013,334.429993,1990100.0,NaN,NaN,NaN,NaN


### 2.2 Size & Value 

### 2.3 Sentiment

## 3. Panel Export

Validate schema, run sanity checks, and write the panel to parquet for research and backtest consumption.

In [6]:
panel.head()
panel.to_parquet(OUTPUT_PATH, index=False)
print(f"saved: {OUTPUT_PATH}")
print(f"shape: {panel.shape}")

saved: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_factor_panel.parquet
shape: (164199, 11)
